# Bootstrapping, IR Derivatives
이번 세션에서는 이자율 기간구조를 산출하는 Bootstrapping 과정과, 간단한 이자율 파생상품에 대해 공부합니다. 

In [1]:
import numpy as np
import pandas as pd
from scipy import optimize
import matplotlib.pyplot as plt

## 1. Term Structure
시작하기에 앞서 이자율 기간구조에 대해 알아보겠습니다. 이자율이란, 돈을 빌린 대가로 원금에 더해 지급되는 금액의 비율입니다.<br>
이때 이자율은 지급주체의 지급불이행 위험, 즉 신용도와, 지급까지의 기간에 따라 달라지는 양상을 보입니다.<br>
따라서 같은 주체가 발행한 채권이라 하더라도, 만기에 따라 표면이자율이 다른 것이 이러한 현상 때문에 일어납니다.<br>
즉 현금흐름의 지급 시점에 따라서 해당 현금흐름에 단위시간마다 가산/감산되는 가치가 달라진다는 것입니다. <br>
이자율 기간구조는 이를 표현한 곡선으로, 주로 한 주체가 지급할 현금흐름을 할인하여 해당 금융상품의 가격을 책정할 때 사용합니다.<br>
이자율 기간구조로는 YTM Curve와 Zero Curve가 있는데, YTM Curve는 사용이 복잡한 반면, Zero Curve는 해당 만기와 대응되는 이자율을 다음 식에 대입해 쉽게 할인률을 구할 수 있습니다.
$$DF = e^{-rt}$$
따라서 오늘 세션에서는 Zero Curve만을 사용합니다.

## 2. IRS and Risk-Free Rate Term Structure
IRS(Interest Rate Swap)은 가장 단순한 이자율 파생상품으로, 고정 금리와 변동 금리를 주기적으로 교환하는 상품입니다. <br>
IRS는 CCP라고 불리는 청산소에서 증거금을 수취하고, 지급을 보증하기에 무위험 이자율로 사용됩니다.<br>
우리나라에서 IRS는 대개 CD 91일물 금리와 고정 금리를 3개월마다 교환합니다. 이를 이용해 이자율 기간구조를 산출해 보겠습니다.<br>
우선 IRS의 현금흐름 지급 시점은 $0.25 \times N$ 입니다. 고정 금리 지급자는 수취자에게 해당 시점마다 고정된 이자율을 지급합니다. <br>
이때 이자율은 연율로 들어가니 지급액은 $0.25 \times NA \times r_{fixed}$입니다. $NA$는 원금을 의미합니다.<br>
그리고 변동 금리 지급자는, 거래일로부터 $0.25 \times N-1$ 년 후 시점에 고시된 CD91일물 금리를 $r_{N-1}$이라고 하면, <br>
거래일로부터 $0.25 \times N$년 후 시점에  $0.25 \times NA \times r_{N-1}$에 해당하는 금액을 수취자에게 지급합니다. <br>
이때, 미래 시점에 고시될 CD91일물 금리를 첫 시간에 배운 선도이자율을 이용해 추정합니다.<br>
하지만 선도이자율은 이자율 기간구조가 존재할 때 추정이 가능합니다. 하지만 현재 기간구조를 알 수 없기 때문에 해당 과정을 거칩니다.<br>

0. 기간구조는 선형 보간을 전제합니다. 또, 모든 IRS에 대해, 변동 Leg와 고정 Leg의 현재가치는 같습니다.
1. 기간구조에 현재 고시된 CD91일물 금리를 이용해 $T = 0.25$ 에 대응되는 이자율을 대입합니다.
2. 현재 고시된 IRS중 가장 짧은 만기에 대응되는 이자율을 $r$ 이라고 가정합니다.
3. 완성된 기간구조를 이용해 선도이자율을 계산하여 각 시점에서의 지급액을 구하고, 이를 통해 변동 Leg와 고정 Leg의 현재가치를 구합니다.
4. 2~3을 반복하며 두 Leg의 현재가치가 동일해지는 $r$을 구합니다.
5. 그 다음으로 짧은 만기의 IRS에 대해 2~4를 반복합니다.

해당 과정은 IRS의 기준금리와 할인금리가 동일하기에 성립합니다.<br>
우선 만기와 고정금리,  현재까지 확정된 기간구조와, 현재 가정한 마지막 금리를 입력하면 해당 IRS의 현재가치를 구하는 함수를 작성합니다.

In [2]:
def PricingIRS( Last_R, Maturity, FixedRate, Rate_T, Rate_R):
    Price = 0
    N = (int)(Maturity * 4)
    newRate_T = Rate_T.copy()
    newRate_R = Rate_R.copy()
    newRate_T = np.append(newRate_T,Maturity)
    newRate_R = np.append(newRate_R,Last_R)
    for i in range(0, N):
        T1 = i * 0.25
        T2 = (i+1) * 0.25
        if(i == 0):
            DF1 = 1.0
        else:
            DF1 = np.exp(-np.interp(T1, newRate_T, newRate_R) * T1)
        DF2 = np.exp(-np.interp(T2, newRate_T, newRate_R) * T2)
        Forward = ((DF1/DF2)-1)/(T2 - T1)
        CFFloat = Forward * DF2 * 0.25
        CFFixed = FixedRate * DF2 * 0.25
        Price += CFFloat - CFFixed
    return Price

그리고 현재 IRS 금리를 입력받아 IRS 만기에 대응되는 이자율을 구하는 함수를 작성합니다. 해당 과정은 최적화를 이용합니다.

In [3]:
def FindZero( Maturity, FixedRate, Rate_T, Rate_R):
    r = optimize.fsolve(PricingIRS, 0.03, args = (Maturity, FixedRate, Rate_T, Rate_R))[0]
    return r

그리고 기간구조 산출에 사용할 IRS 금리를 받아옵니다.

In [4]:
IRS_T = np.array([1, 2, 3, 4, 5, 7, 10, 12, 15, 20])
IRS_R = np.array([0.025750, 0.026075, 0.026675, 0.0273, 0.02785, 0.028525, 0.02915, 0.029475, 0.029325, 0.028425])
#출처 = http://www.smbs.biz/Exchange/IRS.jsp

우선 현재 CD91일물 금리를 이용해 시점 0.25에 대응되는 이자율을 채웁니다. 현재 CD91일물 금리는 2.56%입니다.

In [5]:
Term_T = np.array([0.25])
Term_R = np.array([0.0255184])

그리고 받아온 IRS 금리를 이용해서 IRS 만기에 대응되는 이자율들을 찾아 기간구조를 완성합니다.

In [6]:
for i in range(0, len(IRS_T)):
    TargetT = IRS_T[i]
    TargetR = IRS_R[i]
    r = FindZero( TargetT, TargetR, Term_T, Term_R)
    Term_T = np.append(Term_T, TargetT)
    Term_R = np.append(Term_R, TargetR)

In [7]:
Term_R

array([0.0255184, 0.02575  , 0.026075 , 0.026675 , 0.0273   , 0.02785  ,
       0.028525 , 0.02915  , 0.029475 , 0.029325 , 0.028425 ])

이런 과정을 통해 BootStrapping을 이용하여, 시장에 존재하는 금융상품으로부터 이자율 커브를 도출할 수 있습니다.<br>
이런 이자율커브를 이용하여 현재가치를 계산하는 것으로 옵션성이 존재하지 않는 이자율 상품(일반채권, IRS...)를 평가할 수 있습니다.

## 3. Hull-White Model and Cap Pricing
옵션성이 존재하는 상품의 경우, 해당 옵션의 행사 여부를 판단하기 위해 주식 파생과 같이 이자율의 시나리오를 산출하는 과정을 진행해야 합니다.<br>
Hull White 모형은 이러한 이자율 시나리오 작성에 사용하는 모형으로, 특정 시점 $T$와 그로부터 미소 시간 이후인 $T+dt$ 사이의 구간에 유효한 Short Rate를<br>
모델링하는 모형입니다. Hull White 모형의 식은 다음과 같습니다.
$$dr(t) = [\theta(t) - \alpha r(t)]dt + \sigma (t) dW$$
즉 해당 모형에서 이자율의 변화는 장기 평균을 향해 가는 평균회귀성과 변동성의 합으로 주어집니다. 이때 $\alpha$와 $\sigma$는 주로 스왑션 시장에서 도출합니다. <br>
본 세션에서는 계산의 편의를 위해 다음과 같은 가정하에 평가를 진행합니다.<br>
$$\theta (t) = \theta = 0.03$$
$$r(0) = 0.03$$
$$\alpha = 0.01$$
$$\sigma(t) = \sigma = 0.005$$
그리고 해당 모형에서 모델링하는 것은 short rate이기 때문에 이 short rate로 만들어진 short 할인계수를 곱해서 두 시점 사이의 할인계수를 만드는 함수를 제작합니다.

In [8]:
def CalcDF(shortrate, start, end, dt):
    DF = 1.0
    for i in range(start,end):
        r = shortrate[i]
        shortDF = np.exp(-r*dt)
        DF *= shortDF
    return DF

이 함수를 통해 만든 DF를 이용해서 선도금리를 계산하고, 이를 이용해 옵션 행사 여부와 행사시 이득을 계산합니다.<br>
이제 해당 모형을 이용해 다음과 같은 금리캡(Cap)의 가치를 계산해 보겠습니다.<br>
$$Payoff = NA \times max(r_{CD}(T) - r_{strike})$$
$$T = 1$$
$$NA = 10000$$
이때, $r_{CD}$는 CD91일물 금리를 의미합니다. 만기시점에서 CD91일물 금리를 구하기 위해 DF를 이용해 선도금리를 계산합니다.<br>
우선 시뮬레이션을 실행합니다.

In [9]:
def next_shortrate(rt, kappa, sigma, theta, dt, dW):
    drift = kappa*(theta - rt)*dt
    rand = sigma * np.sqrt(dt) * dW
    return rt + drift + rand

NTrial = 10000
NA = 10000
kappa = 0.01
HWVol = 0.005
theta = 0.03
Strike = 0.03
rf = 0.03
dt = 1/365
T = 1

In [10]:
ShortRate = pd.DataFrame(index = range(1,NTrial+1), columns = range(0,int(T/dt)+1+91)) #CD91일물 추정을 위해 만기 91일 후까지 제작
ShortRate[0] = rf
RfRand = np.random.normal(size=(int(T/dt) + 91, NTrial))
for i in range(0,int(T/dt)+91):
    ShortRate[i+1] = next_shortrate(ShortRate[i], kappa, HWVol, theta, dt, RfRand[i])

이제 다음 식을 이용해서 만기에서의 CD91일물 금리를 구합니다.
$$f = {{{{DF_{1}}\over{DF_{2}}}-1}\over{T2-T1}}$$
이 상품에서의 $T1 = 1$이고 $T2 = 1+{91\over365}$가 됩니다.

In [11]:
CD = np.zeros(NTrial)
Payoff = np.zeros(NTrial)
for i in range(0,NTrial):
    DF1 = CalcDF(ShortRate.iloc[i], 365, 365+91, dt)
    CDRate = ((1/DF1)-1)/(91/365)
    CD[i] = CDRate
    Payoff[i] = np.maximum(CDRate - Strike,0) * NA

이제 도출한 Payoff들의 현재가치를 구하면 금리캡의 가치를 구할 수 있습니다.

In [12]:
Price = 0.0
for i in range(0,NTrial):
    Price += Payoff[i] * CalcDF(ShortRate.iloc[i], 0, 365, dt)
Price/=10000
print("Cap의 가격은 %f 입니다"%Price)

Cap의 가격은 20.661756 입니다


## 4. Pricing Callable Bond with LSMC
이자율 파생상품에서도 LSMC를 사용해야 하는 경우가 존재합니다.<br>
만기 이전에 옵션을 행사할 수 있는 경우에 LSMC를 사용하는데 이러한 형태의 옵션을 가진 가장 간단한 상품은 Callable Bond입니다.<br>
Callable Bond는 일반채권에 콜옵션이 달린 상품으로, 채권자는 일정 시기마다 행사가를 지불하고 채권을 되사올 권리가 있습니다.<br>
아메리칸 옵션과 같이, 보유가치와 행사가치를 비교하는데, 모든 Path에 대해 LSMC를 진행하며, 주가가 아닌 해당 시점의 Short Rate를 사용하여 LSMC를 진행합니다.<br>
발행정보는 다음과 같습니다.<br>
$$NA = 10000$$
$$coupon = 0.024$$
$$frequency_{coupon} = 1/12$$
$$frequency_{calloption} = 1/12 $$
$$K = 10000$$

LSMC에서 보유가치는 미래에 있을 원금과 쿠폰 현금흐름의 현재가치입니다. 그리고 행사가치는 행사가입니다.<br>
보유가치를 선형회귀분석으로 추정해, 보유가치가 행사가치보다 큰 경우 옵션을 행사합니다.<br>

In [13]:
ContiValue = pd.DataFrame(index = range(1,NTrial+1), columns = range(1,13))
ContiValue[12] = 10020

옵션 행사시점은 1월부터 11월까지 각 달의 마지막 날로 지정합니다.<br>

In [14]:
OptionT = [31,59,90,120,151,181,212,243,273,304,334,366] #만기에는 옵션 없음
for i in range(1,12):
    T1 = OptionT[i-1]-1 # 현재 시점이 1이라고 생각해서 사이 간격은 1을 뺌
    T2 = OptionT[i]-1
    ContiValue[12-i] = ContiValue[12-i+1] * CalcDF(ShortRate, T1, T2, dt) + 20
OriginalPrice = np.mean(ContiValue[1]* CalcDF(ShortRate, 0, 30, dt))

이제 보유가치의 추정치와 행사가치를 비교하여, 행사가치가 보유가치의 추정치보다 작은 경우, 해당 path,  해당 시점의 가치를 행사가치로 바꿉니다.

In [15]:

for i in range(1,12):
    X = np.zeros([NTrial,3])
    y = np.array(ContiValue[12-i]) 
    X[:,0]+=1
    X[:,1]+=ShortRate[OptionT[i]-1]
    X[:,2]+=ShortRate[OptionT[i]-1]**2
    beta = np.linalg.inv(X.T@X)@X.T@y
    EY = X@beta
    ContiValue[12-i] += (EY>10000) * (10000-ContiValue[12-i])


print("Callable Bond의 가격은 %f 입니다"%np.mean(ContiValue[1]* CalcDF(ShortRate, 0, 30, dt)))
print("Non - Callable Bond의 가격은 %f 입니다"%OriginalPrice)
print("CallOption의 가격은 %f입니다"%(np.mean(ContiValue[1]* CalcDF(ShortRate, 0, 30, dt)) - OriginalPrice))

Callable Bond의 가격은 9939.829237 입니다
Non - Callable Bond의 가격은 9940.645406 입니다
CallOption의 가격은 -0.816169입니다


발행자에게 주어진 옵션이기에 투자자 입장에서 진행하는 가치평가의 특성상 음수의 가격이 나오는 것을 확인할 수 있습니다.